In [1]:
import os
import pandas as pd

# Local data folder (same directory as this notebook)
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
sample_size = 1

df_r = pd.read_csv(os.path.join(DATA_DIR, 'RESULTADOS_2024.csv'), encoding='latin1',sep=";")
df_r_sample = df_r.sample(frac=sample_size, random_state=42)
df_e = pd.read_csv(os.path.join(DATA_DIR, 'microdados_ed_basica_2024.csv'), encoding='latin1',sep=";",low_memory=False)

In [3]:
school_columns = [
    "NO_REGIAO",
    "SG_UF",
    "NO_MUNICIPIO",
    "CO_ENTIDADE",
    "TP_DEPENDENCIA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO",
    "IN_VINCULO_SECRETARIA_EDUCACAO",
    "IN_VINCULO_SEGURANCA_PUBLICA",
    "IN_VINCULO_SECRETARIA_SAUDE",
    "IN_VINCULO_OUTRO_ORGAO",
    "IN_MANT_ESCOLA_PRIVADA_EMP",
    "IN_MANT_ESCOLA_PRIVADA_ONG",
    "IN_MANT_ESCOLA_PRIVADA_OSCIP",
    "IN_MANT_ESCOLA_PRIV_ONG_OSCIP",
    "IN_MANT_ESCOLA_PRIVADA_SIND",
    "IN_MANT_ESCOLA_PRIVADA_SIST_S",
    "IN_MANT_ESCOLA_PRIVADA_S_FINS",
    "IN_LOCAL_FUNC_PREDIO_ESCOLAR",
    "IN_LOCAL_FUNC_SOCIOEDUCATIVO",
    "IN_LOCAL_FUNC_UNID_PRISIONAL",
    "IN_LOCAL_FUNC_PRISIONAL_SOCIO",
    "IN_LOCAL_FUNC_GALPAO",
    "IN_LOCAL_FUNC_SALAS_OUTRA_ESC",
    "IN_LOCAL_FUNC_OUTROS",
    "IN_PREDIO_COMPARTILHADO",
    "IN_AGUA_INEXISTENTE",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_INEXISTENTE",
    "IN_TRATAMENTO_LIXO_INEXISTENTE",
    "IN_AREA_VERDE",
    "IN_AUDITORIO",
    "IN_BANHEIRO",
    "IN_BIBLIOTECA",
    "IN_DORMITORIO_ALUNO",
    "IN_DORMITORIO_PROFESSOR",
    "IN_LABORATORIO_CIENCIAS",
    "IN_LABORATORIO_INFORMATICA",
    "IN_PATIO_COBERTO",
    "IN_PATIO_DESCOBERTO",
    "IN_PISCINA",
    "IN_QUADRA_ESPORTES",
    "IN_REFEITORIO",
    "IN_SALA_ATELIE_ARTES",
    "IN_SALA_MUSICA_CORAL",
    "IN_SALA_ESTUDIO_DANCA",
    "IN_SALA_MULTIUSO",
    "IN_SALA_ESTUDIO_GRAVACAO",
    "IN_SALA_OFICINAS_EDUC_PROF",
    "IN_SALA_LEITURA",
    "IN_SALA_PROFESSOR",
    "IN_SALA_REPOUSO_ALUNO",
    "IN_SECRETARIA",
    "IN_ACESSIBILIDADE_INEXISTENTE",
    "QT_SALAS_UTILIZADAS",
    "QT_SALAS_UTILIZA_CLIMATIZADAS",
    "IN_COMPUTADOR",
    "IN_EQUIP_IMPRESSORA",
    "IN_EQUIP_SOM",
    "IN_EQUIP_TV",
    "IN_EQUIP_LOUSA_DIGITAL",
    "IN_EQUIP_MULTIMIDIA",
    "IN_DESKTOP_ALUNO",
    "IN_COMP_PORTATIL_ALUNO",
    "IN_TABLET_ALUNO",
    "IN_INTERNET",
    "IN_INTERNET_ALUNOS",
    "IN_PROF_BIBLIOTECARIO",
    "IN_PROF_SAUDE",
    "IN_PROF_COORDENADOR",
    "IN_PROF_FONAUDIOLOGO",
    "IN_PROF_NUTRICIONISTA",
    "IN_PROF_PSICOLOGO",
    "IN_PROF_ALIMENTACAO",
    "IN_PROF_PEDAGOGIA",
    "IN_PROF_SECRETARIO",
    "IN_PROF_SEGURANCA",
    "IN_PROF_MONITORES",
    "IN_PROF_GESTAO",
    "IN_PROF_ASSIST_SOCIAL",
    "IN_ALIMENTACAO",
    "IN_EXAME_SELECAO",
    "IN_RESERVA_NENHUMA",
    "IN_REDES_SOCIAIS",
    "IN_ORGAO_ASS_PAIS",
    "IN_ORGAO_GREMIO_ESTUDANTIL",
    "IN_EDUC_AMBIENTAL",
    "IN_DIURNO",
    "IN_NOTURNO",
    "IN_EAD",
    "IN_PROF",
    "IN_EJA",
    "QT_MAT_EJA_MED",
    "QT_MAT_MED",
    "IN_MATERIAL_PED_NENHUM",
    "QT_DOC_MED",
    "QT_TUR_MED",
    "QT_MAT_PROF",
    "QT_DOC_PROF"
]


df_schools= df_e[school_columns]

result_columns = [
    "NU_SEQUENCIAL",
    "CO_ESCOLA",
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT"
]

df_results = df_r_sample[result_columns]


In [4]:
numeric_columns = df_schools.select_dtypes(include=["number"]).columns
df_schools[numeric_columns] = df_schools[numeric_columns].astype("Int64")

numeric_columns = df_results.select_dtypes(include=["number"]).columns
df_results[numeric_columns] = df_results[numeric_columns].astype("Int64")

In [5]:
nonexistent_columns = [c for c in df_schools.columns if "_INEXISTENTE" in c]

for col in nonexistent_columns:
    new_name = col.replace("_INEXISTENTE", "")

    df_schools[new_name] = 1 - df_schools[col]

df_schools = df_schools.drop(columns=nonexistent_columns)

nonexistent_columns = [c for c in df_schools.columns if "_NENHUM" in c]

for col in nonexistent_columns:
    new_name = col.replace("_NENHUM", "")

    df_schools[new_name] = 1 - df_schools[col]

df_schools = df_schools.drop(columns=nonexistent_columns)

In [ ]:
presence_columns = ["TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT"]

df_results["PRESENCA_COMPLETA"] = df_results[presence_columns].apply(
    lambda row: 1 if all(row != 0) else 0, axis=1
)

public_link_columns= [
    "IN_VINCULO_SECRETARIA_EDUCACAO",
    "IN_VINCULO_SEGURANCA_PUBLICA",
    "IN_VINCULO_SECRETARIA_SAUDE",
    "IN_VINCULO_OUTRO_ORGAO"
]

df_schools["VINCULO_PUBLICO"] = df_schools[public_link_columns].astype(str).agg("".join, axis=1)

private_link_columns = [
    "IN_MANT_ESCOLA_PRIVADA_EMP",
    "IN_MANT_ESCOLA_PRIVADA_ONG",
    "IN_MANT_ESCOLA_PRIVADA_OSCIP",
    "IN_MANT_ESCOLA_PRIV_ONG_OSCIP",
    "IN_MANT_ESCOLA_PRIVADA_SIND",
    "IN_MANT_ESCOLA_PRIVADA_SIST_S",
    "IN_MANT_ESCOLA_PRIVADA_S_FINS"
]

df_schools["VINCULO_PRIVADO"] = df_schools[private_link_columns].astype(str).agg("".join, axis=1)

df_schools = df_schools.drop(columns=public_link_columns + private_link_columns)

operating_location_columns = [
    "IN_LOCAL_FUNC_PREDIO_ESCOLAR",
    "IN_LOCAL_FUNC_SOCIOEDUCATIVO",
    "IN_LOCAL_FUNC_UNID_PRISIONAL",
    "IN_LOCAL_FUNC_PRISIONAL_SOCIO",
    "IN_LOCAL_FUNC_GALPAO",
    "IN_LOCAL_FUNC_SALAS_OUTRA_ESC",
    "IN_LOCAL_FUNC_OUTROS"
]

df_schools["LOCAL_FUNC"] = df_schools[operating_location_columns].astype(str).agg("".join, axis=1)

df_schools = df_schools.drop(columns=operating_location_columns)

# 1. Average high school enrollments per high school class
df_schools["MAT_MED_POR_TURMA"] = df_schools["QT_MAT_MED"] / df_schools["QT_TUR_MED"]

# 2. High school enrollments per classroom in use
df_schools["MAT_MED_POR_SALA"] = df_schools["QT_MAT_MED"] / df_schools["QT_SALAS_UTILIZADAS"]

# 3. Proportion of air-conditioned classrooms
df_schools["PROP_SALAS_CLIMATIZADAS"] = df_schools["QT_SALAS_UTILIZA_CLIMATIZADAS"] / df_schools["QT_SALAS_UTILIZADAS"]

df_schools["MAT_MED_POR_DOC_MED"] = df_schools["QT_MAT_MED"] / df_schools["QT_DOC_MED"]

# 5. High school teachers per high school class
df_schools["DOC_MED_POR_TURMA_MED"] = df_schools["QT_DOC_MED"] / df_schools["QT_TUR_MED"]

float_columns = df_schools.select_dtypes(include=["float"]).columns
df_schools[float_columns] = df_schools[float_columns].round(2)









In [ ]:
df_merge = pd.merge(df_results,
    df_schools,
    left_on="CO_ESCOLA",
    right_on="CO_ENTIDADE",
    how="inner"
)
df_merge = df_merge.dropna(subset=["CO_ESCOLA"])

df_merge.head()


In [ ]:
mean_columns = [
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT",
    "PRESENCA_COMPLETA"
]

data_set = (
    df_merge.groupby("CO_ESCOLA")
    .agg({**{c: "first" for c in df_merge.columns if c not in mean_columns + ["CO_ESCOLA"]},
          **{c: "mean" for c in mean_columns}})
    .reset_index()
)

data_set.drop(columns="CO_ENTIDADE")


data_set

In [ ]:
data_set.to_csv(os.path.join(DATA_DIR, 'data_set.csv'), index=False)